In [1]:
import pandas as pd
import sqlite3
import os

os.makedirs("../data/processed", exist_ok=True)

# Connexion SQLite
conn = sqlite3.connect("../data/processed/market_data.db")

# Charger les CSV
fx_df    = pd.read_csv("../data/raw/fx_data.csv", index_col="date", parse_dates=True)
eq_df    = pd.read_csv("../data/raw/equities_data.csv", index_col="date", parse_dates=True)
rates_df = pd.read_csv("../data/raw/rates_data.csv", index_col="date", parse_dates=True)

print("FX shape      :", fx_df.shape)
print("Equities shape:", eq_df.shape)
print("Rates shape   :", rates_df.shape)


FX shape      : (1564, 2)
Equities shape: (1509, 12)
Rates shape   : (1509, 3)


In [2]:
# Nettoyage : supprimer les valeurs manquantes
fx_clean    = fx_df.dropna()
eq_clean    = eq_df.dropna()
rates_clean = rates_df.dropna()

# Vérification
print("FX après nettoyage      :", fx_clean.shape)
print("Equities après nettoyage:", eq_clean.shape)
print("Rates après nettoyage   :", rates_clean.shape)

# Vérifier qu'il n'y a plus de NaN
print("\nNaN restants FX     :", fx_clean.isna().sum().sum())
print("NaN restants Equities:", eq_clean.isna().sum().sum())
print("NaN restants Rates   :", rates_clean.isna().sum().sum())


FX après nettoyage      : (1564, 2)
Equities après nettoyage: (1509, 12)
Rates après nettoyage   : (1509, 3)

NaN restants FX     : 0
NaN restants Equities: 0
NaN restants Rates   : 0


In [3]:
# Sauvegarder dans la base SQL
fx_clean.to_sql("fx_daily", conn, if_exists="replace", index=True)
eq_clean.to_sql("equities_daily", conn, if_exists="replace", index=True)
rates_clean.to_sql("rates_daily", conn, if_exists="replace", index=True)

print("✅ Tables créées dans market_data.db")


✅ Tables créées dans market_data.db


In [4]:
# Lire directement depuis SQL pour confirmer
df_check = pd.read_sql("SELECT * FROM fx_daily LIMIT 5", conn)
print(df_check)


                  date    eurusd      usdjpy
0  2019-01-01 00:00:00  1.149306  109.629997
1  2019-01-02 00:00:00  1.146171  109.667999
2  2019-01-03 00:00:00  1.131811  107.441002
3  2019-01-04 00:00:00  1.139108  107.807999
4  2019-01-07 00:00:00  1.141044  108.522003
